# Hybrid quantum PPO for the UAV trajectory + scheduling env (fast trainer)

Notebook version of `training_quantumPPO_fast.py`.

- **Actor**: obs $(2M+4)$ → classical pre-encoder → $M$ angles → PQC on $M$ qubits → $\langle Z_q\rangle$ → categorical head over the $M$ users ($m^*$) and Gaussian head for $(\theta_{raw}, v_{raw})$
- **Critic**: obs → pre-encoder → PQC (own parameters) → $\langle Z_q\rangle$ → value head

The PQC uses RY angle encoding, then `n_layers_vqc` layers of RX, RZ on every qubit followed by a CNOT ring. Its gradients come from the parameter-shift rule wired into torch autograd, so both the circuit parameters and the pre-encoder are trained from the PPO loss.

**Fast** means: `n_envs` parallel env copies, so every rollout step is one batched Aer job, and a stochastic parameter shift (`pqc_grad_samples` rows per minibatch).

Circuits run on Qiskit Aer, either **ideal** (statevector, exact $\langle Z\rangle$) or **noisy** (depolarizing + readout noise, or a fake IBM device), with $\langle Z\rangle$ from shots.

`training_quantumPPO_hardware.ipynb` loads this notebook's code (sections 1 to 7) and swaps only the circuit back-end for a real IBM QPU.

## 1. Imports

In [ ]:
%matplotlib inline
import os
import csv
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

### Environment
Loaded from `UAV_traj_env.py` if it sits next to this notebook, otherwise from `UAV_traj_env.ipynb` (its demo section is skipped).

In [ ]:
if os.path.exists("UAV_traj_env.py"):
    from UAV_traj_env import SystemConfig, UAVTrajectoryEnv
else:
    _SKIP_ENV_DEMO = True
    get_ipython().run_line_magic("run", "-i UAV_traj_env.ipynb")
print("env loaded:", UAVTrajectoryEnv.__name__)

## 2. Training configuration

In [ ]:
NN_HIDDEN = 32                       # classical pre-encoder hidden width

TRAIN_CFG = {
    # --- from the project config ---
    "gamma": 0.999,
    "gae_lambda": 0.95,
    "clip_eps": 0.2,
    "lr_classical": 3e-4,
    "lr_quantum": 1e-2,              # VQC parameters usually need a larger learning rate
    "n_layers_vqc": 3,
    "rollout_steps": 1700,           # 170 episodes of N=10 slots per update
    "n_episodes": 1500,
    # --- PPO details ---
    "policy_epochs": 3,              # few epochs: every epoch re-runs all circuits
    "value_epochs": 3,
    "n_envs": 34,                    # parallel env copies: one batched Aer job per step for all of them
    "pqc_grad_samples": 8,           # parameter shift on a random subset of each minibatch (None = all)
    "mini_batch_size": 64,
    "target_kl": 0.02,
    "max_grad_norm": 0.5,
    "ent_coef_user": 0.01,           # entropy bonus, categorical head
    "ent_coef_move": 0.001,          # entropy bonus, Gaussian head
    "init_log_std": -0.5,
    "normalize_returns": True,       # critic learns returns / running std
    # --- Qiskit Aer simulation ---
    "noise": False,                  # True: noisy Aer simulation
    "shots": 1024,                   # shots per circuit when noise=True (0 = exact noisy, no readout error)
    "p1": 1e-3,                      # 1-qubit gate depolarizing probability
    "p2": 1e-2,                      # CNOT depolarizing probability
    "p_meas": 2e-2,                  # readout bit-flip probability
    "noise_backend": None,           # e.g. "FakeManilaV2": noise of a real IBM device
    "resample_users": False,         # True: new user positions every episode
    "resample_demands": False,
    "avg_window": 30,
    "plot_every": 5,                 # save plots, CSVs and checkpoint every N updates
    "seed": 42,
    "out_dir": "results",
}

## 3. Parameterized quantum circuit (PQC)

### 3.1 Ansatz
RY encoding of the $M$ input angles, then `n_layers` × (RX, RZ on every qubit + CNOT ring).

In [ ]:
def build_pqc(n_qubits, n_layers):
    enc_params = ParameterVector("x", n_qubits)
    var_params = ParameterVector("theta", 2 * n_qubits * n_layers)
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(enc_params[i], i)
    qc.barrier(label="encode")
    p = 0
    for layer in range(n_layers):
        for i in range(n_qubits):
            qc.rx(var_params[p], i)
            qc.rz(var_params[p + 1], i)
            p += 2
        for i in range(n_qubits):
            qc.cx(i, (i + 1) % n_qubits)
        if layer < n_layers - 1:
            qc.barrier(label=f"L{layer + 1}")
    return qc, enc_params, var_params

In [ ]:
# Draw the ansatz used in this run
_qc, _, _ = build_pqc(SystemConfig().M, TRAIN_CFG["n_layers_vqc"])
try:
    display(_qc.draw("mpl", fold=-1))
except Exception:          # "mpl" needs pylatexenc; fall back to text
    print(_qc.draw("text", fold=-1))

### 3.2 Noise model
Depolarizing error on every gate + symmetric readout error.

In [ ]:
def build_noise_model(p1, p2, p_meas):
    nm = NoiseModel()
    if p1 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["rx", "ry", "rz"])
    if p2 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])
    if p_meas > 0:
        nm.add_all_qubit_readout_error(ReadoutError([[1 - p_meas, p_meas], [p_meas, 1 - p_meas]]))
    return nm

### 3.3 Aer simulator wrapper
$\langle Z_q\rangle$ of the ansatz for a batch of (encoding, θ) rows, one Aer experiment per row.

- `noise=False`: statevector, exact probabilities
- `noise=True`: noise model, $\langle Z\rangle$ from `shots` (`shots=0`: exact noisy density matrix, no readout error)
- `noise_backend="FakeManilaV2"` etc.: noise of a real IBM device (needs `qiskit-ibm-runtime`)

Qubit $q$ is bit $q$ of the basis-state index (Qiskit little-endian order).

In [ ]:
class PQCSimulator:
    def __init__(self, n_qubits, n_layers, opts=None):
        o = opts or {}
        self.n = n_qubits
        self.L = n_layers
        self.n_params = 2 * n_qubits * n_layers
        idx = np.arange(2 ** n_qubits)
        self.z_sign = np.stack([1.0 - 2.0 * ((idx >> q) & 1) for q in range(n_qubits)], axis=1)  # (dim, n)

        self.noise = bool(o.get("noise", False))
        self.shots = int(o.get("shots", 1024)) if self.noise else 0
        self._seed = int(o.get("seed", 0))
        self.grad_samples = o.get("pqc_grad_samples")
        self.rng = np.random.default_rng(self._seed)
        qc, self.enc_params, self.var_params = build_pqc(n_qubits, n_layers)

        fake_name = o.get("noise_backend") if self.noise else None
        if fake_name:
            from qiskit_ibm_runtime import fake_provider
            fake = getattr(fake_provider, fake_name)()
            self.sim = AerSimulator.from_backend(fake)
            self.shots = max(self.shots, 1)               # device noise -> sample shots
            target = fake
        elif self.noise:
            nm = build_noise_model(o.get("p1", 1e-3), o.get("p2", 1e-2), o.get("p_meas", 2e-2))
            method = "density_matrix" if self.shots == 0 else "automatic"
            self.sim = AerSimulator(method=method, noise_model=nm, max_parallel_experiments=0)
            target = self.sim
        else:
            self.sim = AerSimulator(method="statevector", max_parallel_experiments=0)
            target = self.sim

        if self.shots > 0:
            qc.measure_all()
        else:
            qc.save_probabilities(label="probs")
        opt_level = 1 if fake_name else 0                 # level 0 keeps every gate (and its noise)
        self.tqc = transpile(qc, target, optimization_level=opt_level, seed_transpiler=0)
        self.all_params = list(self.enc_params) + list(self.var_params)

    def expectations(self, x, theta):
        """x: (B, n); theta: (P,) shared or (B, P) per row. Returns (B, n)."""
        x = np.atleast_2d(np.asarray(x, dtype=np.float64))
        theta = np.asarray(theta, dtype=np.float64)
        if theta.ndim == 1:
            theta = np.broadcast_to(theta, (x.shape[0], theta.size))
        rows = np.concatenate([x, theta], axis=1)        # (B, n + P), one experiment per row
        binds = {p: rows[:, i].tolist() for i, p in enumerate(self.all_params)}
        self._seed += 1
        res = self.sim.run(self.tqc, parameter_binds=[binds], shots=max(self.shots, 1),
                           seed_simulator=self._seed).result()
        B = rows.shape[0]
        probs = np.zeros((B, 2 ** self.n))
        for b in range(B):
            if self.shots > 0:
                for key, cnt in res.get_counts(b).items():
                    probs[b, int(key.replace(" ", ""), 2)] += cnt
                probs[b] /= self.shots
            else:
                probs[b] = np.asarray(res.data(b)["probs"], dtype=np.float64)
        return probs @ self.z_sign

### 3.4 Parameter-shift gradients in torch autograd
$\partial\langle Z\rangle/\partial\theta_i = \tfrac12\left[f(\theta_i + \tfrac{\pi}{2}) - f(\theta_i - \tfrac{\pi}{2})\right]$ for both the circuit parameters and the encoded inputs. All shifted circuits of a backward pass go to Aer as **one** batch. With `pqc_grad_samples = k`, only $k$ random rows of the minibatch are shift-evaluated and the gradient is scaled by $B/k$ (unbiased, since the loss gradient is a sum over rows).

In [ ]:
class _PQCFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, theta, sim):
        x_np = x.detach().cpu().double().numpy()
        th_np = theta.detach().cpu().double().numpy()
        ctx.save_for_backward(x, theta)
        ctx.sim = sim
        return torch.as_tensor(sim.expectations(x_np, th_np), dtype=x.dtype, device=x.device)

    @staticmethod
    def backward(ctx, grad_out):
        x, theta = ctx.saved_tensors
        sim = ctx.sim
        x_full = x.detach().cpu().double().numpy()                   # (B_full, n)
        th_np = theta.detach().cpu().double().numpy()                # (P,)
        g_full = grad_out.detach().cpu().double().numpy()            # (B_full, n)
        B_full = x_full.shape[0]
        k_s = sim.grad_samples
        if k_s and k_s < B_full:
            rows = sim.rng.choice(B_full, size=k_s, replace=False)
            scale = B_full / k_s
        else:
            rows, scale = np.arange(B_full), 1.0
        x_np, g = x_full[rows], g_full[rows] * scale
        B, n = x_np.shape
        P = th_np.size
        shift = np.pi / 2.0
        want_x, want_th = ctx.needs_input_grad[0], ctx.needs_input_grad[1]

        xs, ths = [], []
        if want_th:
            for i in range(P):
                for s in (+shift, -shift):
                    t = np.tile(th_np, (B, 1)); t[:, i] += s
                    xs.append(x_np); ths.append(t)
        if want_x:
            for q in range(n):
                for s in (+shift, -shift):
                    xx = x_np.copy(); xx[:, q] += s
                    xs.append(xx); ths.append(np.tile(th_np, (B, 1)))
        if not xs:
            return None, None, None
        E = sim.expectations(np.concatenate(xs), np.concatenate(ths)).reshape(len(xs), B, n)
        dE = 0.5 * (E[0::2] - E[1::2])                                # (K, B, n)

        grad_theta = grad_x = None
        k = 0
        if want_th:
            grad_theta = np.einsum("kbn,bn->k", dE[:P], g)
            grad_theta = torch.as_tensor(grad_theta, dtype=theta.dtype, device=theta.device)
            k = P
        if want_x:
            grad_x = np.zeros_like(x_full)
            grad_x[rows] = np.einsum("qbn,bn->bq", dE[k:k + n], g)
            grad_x = torch.as_tensor(grad_x, dtype=x.dtype, device=x.device)
        return grad_x, grad_theta, None

### 3.5 Quantum layer and classical pre-encoder

In [ ]:
class QuantumLayer(nn.Module):
    def __init__(self, n_qubits, n_layers, qopts=None):
        super().__init__()
        self.sim = PQCSimulator(n_qubits, n_layers, qopts)
        self.theta = nn.Parameter(torch.randn(self.sim.n_params) * 0.1)

    def forward(self, angles):
        return _PQCFunction.apply(angles, self.theta, self.sim)


def _make_pre_encoder(state_dim, n_qubits):
    net = nn.Sequential(
        nn.Linear(state_dim, NN_HIDDEN), nn.Tanh(),
        nn.Linear(NN_HIDDEN, n_qubits), nn.Tanh(),
    )
    for layer in net:
        if isinstance(layer, nn.Linear):
            nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
            nn.init.zeros_(layer.bias)
    return net

## 4. Actor and critic

In [ ]:
class HybridPQCPolicy(nn.Module):
    """Categorical head over users + Gaussian head for (theta_raw, v_raw)."""

    def __init__(self, state_dim, n_users, cont_dim, n_qubits, n_layers, qopts, init_log_std):
        super().__init__()
        self.pre_encoder = _make_pre_encoder(state_dim, n_qubits)
        self.pqc = QuantumLayer(n_qubits, n_layers, qopts)
        self.user_head = nn.Linear(n_qubits, n_users)
        self.move_head = nn.Linear(n_qubits, cont_dim)
        nn.init.orthogonal_(self.user_head.weight, gain=1.0)
        nn.init.zeros_(self.user_head.bias)
        nn.init.orthogonal_(self.move_head.weight, gain=0.1)
        nn.init.zeros_(self.move_head.bias)
        self.log_std = nn.Parameter(torch.full((cont_dim,), float(init_log_std)))

    def classical_parameters(self):
        return (list(self.pre_encoder.parameters()) + list(self.user_head.parameters())
                + list(self.move_head.parameters()) + [self.log_std])

    def forward(self, obs_t):
        angles = self.pre_encoder(obs_t) * (np.pi / 2.0)
        z = self.pqc(angles)
        return self.user_head(z), self.move_head(z)

    def dist(self, obs_t):
        logits, mean = self(obs_t)
        cat = torch.distributions.Categorical(logits=logits)
        std = torch.exp(self.log_std).expand_as(mean)
        return cat, torch.distributions.Normal(mean, std)


class HybridPQCValue(nn.Module):
    def __init__(self, state_dim, n_qubits, n_layers, qopts):
        super().__init__()
        self.pre_encoder_v = _make_pre_encoder(state_dim, n_qubits)
        self.pqc_v = QuantumLayer(n_qubits, n_layers, qopts)
        self.value_head = nn.Linear(n_qubits, 1)

    def classical_parameters(self):
        return list(self.pre_encoder_v.parameters()) + list(self.value_head.parameters())

    def forward(self, obs_t):
        angles = self.pre_encoder_v(obs_t) * (np.pi / 2.0)
        return self.value_head(self.pqc_v(angles)).squeeze(-1)

Running mean/std used to normalise the critic's return targets.

In [ ]:
class RunningMeanStd:
    def __init__(self):
        self.mean, self.var, self.count = 0.0, 1.0, 1e-4

    def update(self, x):
        x = np.asarray(x, dtype=np.float64)
        b_mean, b_var, b_n = x.mean(), x.var(), x.size
        delta = b_mean - self.mean
        tot = self.count + b_n
        self.mean += delta * b_n / tot
        m2 = self.var * self.count + b_var * b_n + delta ** 2 * self.count * b_n / tot
        self.var, self.count = m2 / tot, tot

    @property
    def std(self):
        return float(np.sqrt(self.var) + 1e-8)

## 5. PPO agent
Action selection (single and batched over the parallel envs), GAE, and the clipped PPO update. Separate Adam optimisers for classical weights (`lr_classical`) and PQC parameters (`lr_quantum`); early stop on `target_kl`.

In [ ]:
class QuantumPPOAgent:
    def __init__(self, obs_dim, n_users, cont_dim, cfg):
        self.cfg = cfg
        n_qubits = n_users                         # M qubits for scheduling
        qopts = {k: cfg[k] for k in ["noise", "shots", "p1", "p2", "p_meas", "noise_backend", "pqc_grad_samples"]}
        qopts_v = dict(qopts, seed=cfg["seed"] * 1000 + 1)
        qopts = dict(qopts, seed=cfg["seed"] * 1000)
        self.actor = HybridPQCPolicy(obs_dim, n_users, cont_dim, n_qubits,
                                     cfg["n_layers_vqc"], qopts, cfg["init_log_std"])
        self.value = HybridPQCValue(obs_dim, n_qubits, cfg["n_layers_vqc"], qopts_v)

        self.actor_nn_opt = optim.Adam(self.actor.classical_parameters(), lr=cfg["lr_classical"])
        self.value_nn_opt = optim.Adam(self.value.classical_parameters(), lr=cfg["lr_classical"])
        self.actor_q_opt = optim.Adam([self.actor.pqc.theta], lr=cfg["lr_quantum"])
        self.value_q_opt = optim.Adam([self.value.pqc_v.theta], lr=cfg["lr_quantum"])
        self.ret_rms = RunningMeanStd()

    def _value(self, obs_t):
        """Value in real return units."""
        v = self.value(obs_t)
        return v * self.ret_rms.std if self.cfg["normalize_returns"] else v

    @torch.no_grad()
    def select_action(self, obs, deterministic=False):
        obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
        cat, gauss = self.actor.dist(obs_t)
        if deterministic:
            m, cont = cat.probs.argmax(-1), gauss.mean
        else:
            m, cont = cat.sample(), gauss.sample()
        logp = cat.log_prob(m) + gauss.log_prob(cont).sum(-1)
        value = self._value(obs_t)
        return int(m.item()), cont[0].numpy(), float(logp.item()), float(value.item())

    @torch.no_grad()
    def select_actions(self, obs_batch):
        """Batched over the E parallel envs: one Aer job for the actor, one for the critic."""
        obs_t = torch.as_tensor(np.asarray(obs_batch), dtype=torch.float32)
        cat, gauss = self.actor.dist(obs_t)
        m, cont = cat.sample(), gauss.sample()
        logp = cat.log_prob(m) + gauss.log_prob(cont).sum(-1)
        value = self._value(obs_t)
        return m.numpy(), cont.numpy(), logp.numpy(), value.numpy()

    @torch.no_grad()
    def values_of(self, obs_batch):
        return self._value(torch.as_tensor(np.asarray(obs_batch), dtype=torch.float32)).numpy()

    def _mini_batches(self, length):
        idx = np.random.permutation(length)
        bs = self.cfg["mini_batch_size"]
        for s in range(0, length, bs):
            yield idx[s:s + bs]

    def compute_gae(self, rewards, values, dones, last_value):
        gamma, lam = self.cfg["gamma"], self.cfg["gae_lambda"]
        T = len(rewards)
        adv = np.zeros(np.shape(rewards), dtype=np.float64)    # (T, E)
        gae, next_v = 0.0, np.asarray(last_value, dtype=np.float64)
        for t in reversed(range(T)):
            nonterm = 1.0 - dones[t]
            delta = rewards[t] + gamma * next_v * nonterm - values[t]
            gae = delta + gamma * lam * nonterm * gae
            adv[t] = gae
            next_v = values[t]
        return adv, adv + np.asarray(values)

    def update(self, buf, last_value):
        c = self.cfg
        # buffers are (T, E, ...): GAE along time per env, then flatten
        adv, ret = self.compute_gae(np.array(buf["rewards"]), np.array(buf["values"]),
                                    np.array(buf["dones"]), last_value)
        adv, ret = adv.reshape(-1), ret.reshape(-1)
        obs_np = np.array(buf["obs"])
        obs_t = torch.as_tensor(obs_np.reshape(-1, obs_np.shape[-1]), dtype=torch.float32)
        m_t = torch.as_tensor(np.array(buf["users"]).reshape(-1), dtype=torch.long)
        cont_np = np.array(buf["conts"])
        cont_t = torch.as_tensor(cont_np.reshape(-1, cont_np.shape[-1]), dtype=torch.float32)
        logp_old = torch.as_tensor(np.array(buf["logp"]).reshape(-1), dtype=torch.float32)
        if c["normalize_returns"]:
            self.ret_rms.update(ret)
            ret_target = ret / self.ret_rms.std
        else:
            ret_target = ret
        adv_t = torch.as_tensor(adv, dtype=torch.float32)
        adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)
        ret_t = torch.as_tensor(ret_target, dtype=torch.float32)
        T = len(adv)

        a_losses, v_losses, ent_u, ent_m, kls, gq_a, gq_v = [], [], [], [], [], [], []

        # ---------------- actor ----------------
        for _ in range(c["policy_epochs"]):
            for idx in self._mini_batches(T):
                cat, gauss = self.actor.dist(obs_t[idx])
                logp = cat.log_prob(m_t[idx]) + gauss.log_prob(cont_t[idx]).sum(-1)
                e_u = cat.entropy().mean()
                e_m = gauss.entropy().sum(-1).mean()
                ratio = torch.exp(logp - logp_old[idx])
                s1 = ratio * adv_t[idx]
                s2 = torch.clamp(ratio, 1 - c["clip_eps"], 1 + c["clip_eps"]) * adv_t[idx]
                loss = -torch.min(s1, s2).mean() - c["ent_coef_user"] * e_u - c["ent_coef_move"] * e_m

                self.actor_nn_opt.zero_grad()
                self.actor_q_opt.zero_grad()
                loss.backward()                    # PQC grads via parameter shift
                nn.utils.clip_grad_norm_(self.actor.classical_parameters(), c["max_grad_norm"])
                gq_a.append(float(nn.utils.clip_grad_norm_([self.actor.pqc.theta], c["max_grad_norm"])))
                self.actor_nn_opt.step()
                self.actor_q_opt.step()

                a_losses.append(loss.item()); ent_u.append(e_u.item()); ent_m.append(e_m.item())

            with torch.no_grad():
                cat, gauss = self.actor.dist(obs_t)
                logp_new = cat.log_prob(m_t) + gauss.log_prob(cont_t).sum(-1)
                log_r = logp_new - logp_old
                approx_kl = float(((torch.exp(log_r) - 1) - log_r).mean())
            kls.append(approx_kl)
            if approx_kl > c["target_kl"]:
                break

        # ---------------- critic ----------------
        for _ in range(c["value_epochs"]):
            for idx in self._mini_batches(T):
                v_pred = self.value(obs_t[idx])
                v_loss = 0.5 * nn.functional.mse_loss(v_pred, ret_t[idx])
                self.value_nn_opt.zero_grad()
                self.value_q_opt.zero_grad()
                v_loss.backward()
                nn.utils.clip_grad_norm_(self.value.classical_parameters(), c["max_grad_norm"])
                gq_v.append(float(nn.utils.clip_grad_norm_([self.value.pqc_v.theta], c["max_grad_norm"])))
                self.value_nn_opt.step()
                self.value_q_opt.step()
                v_losses.append(v_loss.item())

        return {
            "actor_loss": float(np.mean(a_losses)),
            "critic_loss": float(np.mean(v_losses)),
            "entropy_user": float(np.mean(ent_u)),
            "entropy_move": float(np.mean(ent_m)),
            "kl": kls[-1] if kls else 0.0,
            "actor_theta_grad": float(np.mean(gq_a)),
            "value_theta_grad": float(np.mean(gq_v)),
        }

## 6. Logging, saving and plotting helpers
### 6.1 Console log

In [ ]:
HEADER = (f" {'Upd':>4} | {'Eps':>5} | {'Return':>8} | {'MA Ret':>8} | {'Data Mb':>8} | {'Sat':>4} | "
          f"{'Zone':>4} | {'E left':>6} | {'A Loss':>7} | {'C Loss':>7} | {'H_u':>5} | {'KL':>6} | "
          f"{'gQ_A':>5} | {'gQ_V':>5}")


def print_training_header(env, cfg_train, n_updates):
    c = env.cfg
    qmode = (f"NOISY ({cfg_train['noise_backend'] or 'depolarizing'}, shots={cfg_train['shots']})"
             if cfg_train["noise"] else "ideal")
    print("=" * len(HEADER))
    print(f"Quantum PPO | obs={env.state_dim} | users M={c.M} (qubits) | N={c.N} slots | "
          f"VQC layers={cfg_train['n_layers_vqc']} | Qiskit Aer {qmode} | "
          f"resample_users={c.resample_users} | ~{n_updates} updates")
    print("=" * len(HEADER))
    print(HEADER)
    print("-" * len(HEADER))


def print_training_summary(history):
    k = 50
    print("=" * len(HEADER))
    print("TRAINING SUMMARY (last episodes)")
    print(f"  mean return          : {np.mean(history['ep_return'][-k:]):.3f}")
    print(f"  mean delivered data  : {np.mean(history['ep_data_Mb'][-k:]):.1f} Mb")
    print(f"  users fully served   : {np.mean(history['ep_satisfied'][-k:]):.2f} / {history['M']}")
    print(f"  zone entries / ep    : {np.mean(history['ep_zone'][-k:]):.2f}")
    print(f"  energy left (J)      : {np.mean(history['ep_E_left'][-k:]):.1f}")
    print(f"  final dist to end (m): {np.mean(history['ep_dist_end'][-k:]):.2f}")
    print("=" * len(HEADER))


def _moving_avg(x, w):
    x = np.asarray(x, dtype=np.float64)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")

### 6.2 Save / load results
`episodes.csv` (one row per episode), `updates.csv` (one row per PPO update), `eval_trajectory.csv` (greedy trajectory) and `run_info.json` (configs, users, greedy-eval summary), so runs can be re-plotted and compared with the classical PPO.

In [ ]:
EP_COLS = ["ep_update", "ep_return", "ep_data_Mb", "ep_satisfied", "ep_unmet_frac",
           "ep_zone", "ep_forced", "ep_E_left", "ep_dist_end"]
UPD_COLS = ["actor_loss", "critic_loss", "entropy_user", "entropy_move", "kl",
            "actor_theta_grad", "value_theta_grad"]


def _write_csv(path, header, rows):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


def _jsonable(v):
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, (np.floating, np.integer, np.bool_)):
        return v.item()
    if isinstance(v, (list, tuple)):
        return [_jsonable(x) for x in v]
    if isinstance(v, dict):
        return {k: _jsonable(x) for k, x in v.items()}
    return v


def save_results(history, out_dir, cfg_train, cfg_sys, env=None, eval_summary=None, traj=None, served=None):
    os.makedirs(out_dir, exist_ok=True)
    n_ep = len(history["ep_return"])
    _write_csv(os.path.join(out_dir, "episodes.csv"), ["episode"] + [c[3:] for c in EP_COLS],
               [[i + 1] + [history[c][i] for c in EP_COLS] for i in range(n_ep)])
    n_up = len(history["actor_loss"])
    _write_csv(os.path.join(out_dir, "updates.csv"), ["update"] + UPD_COLS,
               [[i + 1] + [history[c][i] for c in UPD_COLS] for i in range(n_up)])
    if traj is not None:
        traj = np.asarray(traj)
        _write_csv(os.path.join(out_dir, "eval_trajectory.csv"), ["slot", "x", "y", "served_user"],
                   [[n + 1, traj[n, 0], traj[n, 1], served[n] if n < len(served) else ""]
                    for n in range(len(traj))])
    info = {
        "algorithm": "Quantum PPO (" + ("noisy Aer" if cfg_train.get("noise") else "ideal Aer") + ")",
        "train_cfg": cfg_train,
        "sys_cfg": cfg_sys.__dict__,
        "user_xy": env.user_xy if env is not None else None,
        "demands_bits": env.D if env is not None else None,
        "eval_greedy": eval_summary,
        "n_episodes_done": n_ep,
        "n_updates_done": n_up,
    }
    with open(os.path.join(out_dir, "run_info.json"), "w") as f:
        json.dump(_jsonable(info), f, indent=2)


def load_results(out_dir):
    """Returns {'episodes': {col: array}, 'updates': {...}, 'trajectory': {...} or None, 'info': dict}."""
    def read(path):
        if not os.path.exists(path):
            return None
        with open(path) as f:
            rows = list(csv.reader(f))
        cols = {h: [] for h in rows[0]}
        for r in rows[1:]:
            for h, v in zip(rows[0], r):
                cols[h].append(float(v) if v not in ("", None) else np.nan)
        return {h: np.asarray(v) for h, v in cols.items()}
    info_path = os.path.join(out_dir, "run_info.json")
    info = json.load(open(info_path)) if os.path.exists(info_path) else {}
    return {"episodes": read(os.path.join(out_dir, "episodes.csv")),
            "updates": read(os.path.join(out_dir, "updates.csv")),
            "trajectory": read(os.path.join(out_dir, "eval_trajectory.csv")),
            "info": info}

### 6.3 Plots
These save PNGs into `out_dir` and close the figure, so nothing pops up during training. Section 9 shows the saved images.

In [ ]:
def plot_training(history, out_dir, w=30):
    fig, axs = plt.subplots(2, 3, figsize=(15, 8))
    for ax, key, title in [
        (axs[0, 0], "ep_return", "Episode return"),
        (axs[0, 1], "ep_data_Mb", "Delivered data per episode (Mb)"),
        (axs[0, 2], "ep_satisfied", "Users with demand met"),
    ]:
        ax.plot(history[key], alpha=0.3)
        ax.plot(np.arange(len(_moving_avg(history[key], w))) + min(w, len(history[key])) - 1,
                _moving_avg(history[key], w))
        ax.set_title(title); ax.grid(True)
    axs[1, 0].plot(history["actor_loss"], label="actor")
    axs[1, 0].plot(history["critic_loss"], label="critic")
    axs[1, 0].set_title("Losses (per update)"); axs[1, 0].legend(); axs[1, 0].grid(True)
    axs[1, 1].plot(history["entropy_user"], label="user (categorical)")
    axs[1, 1].plot(history["entropy_move"], label="move (Gaussian)")
    axs[1, 1].set_title("Policy entropy"); axs[1, 1].legend(); axs[1, 1].grid(True)
    axs[1, 2].plot(history["actor_theta_grad"], label="actor")
    axs[1, 2].plot(history["value_theta_grad"], label="critic")
    axs[1, 2].set_title("PQC gradient norm (parameter shift)"); axs[1, 2].legend(); axs[1, 2].grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "quantum_ppo_training.png"), dpi=120)
    plt.close()


def plot_trajectory(env, traj, served, out_dir, name="trajectory.png"):
    c = env.cfg
    traj = np.asarray(traj)
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    for (cx, cy), r in zip(env.core.zone_c, env.core.zone_r):
        ax.add_patch(plt.Circle((cx, cy), r, color="red", alpha=0.25))
    colors = plt.cm.tab10(np.arange(c.M) % 10)
    for m, (ux, uy) in enumerate(env.user_xy):
        ax.scatter(ux, uy, marker="^", s=120, color=colors[m], edgecolor="k", zorder=3)
        ax.annotate(f"U{m}", (ux + 6, uy + 6))
    ax.plot(traj[:, 0], traj[:, 1], "-", color="gray", lw=1)
    ax.scatter(traj[:-1, 0], traj[:-1, 1], c=[colors[m] for m in served], s=18, zorder=4)
    # label each slot; slots spent at the same spot (hovering) share one label
    groups = {}
    for n, (x, y) in enumerate(traj[:-1], start=1):
        groups.setdefault((round(x, 1), round(y, 1)), []).append(n)
    for (x, y), slots in groups.items():
        if len(slots) == 1:
            txt = f"{slots[0]}"
        elif slots == list(range(slots[0], slots[-1] + 1)):
            txt = f"{slots[0]}-{slots[-1]}"
        else:
            txt = ",".join(map(str, slots))
        ax.annotate(txt, (x, y), xytext=(4, -10), textcoords="offset points", fontsize=7, color="dimgray")
    ax.scatter(*c.start, marker="s", s=100, color="green", label="start")
    ax.scatter(*c.end, marker="*", s=180, color="gold", edgecolor="k", label="end")
    ax.set_xlim(-10, c.area + 10); ax.set_ylim(-10, c.area + 10); ax.set_aspect("equal")
    ax.set_title("UAV trajectory (dot colour = served user)"); ax.legend(loc="upper left"); ax.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, name), dpi=120)
    plt.close()

### 6.4 Greedy evaluation

In [ ]:
def evaluate(agent, env, seed=None):
    """One greedy episode. Returns (summary dict, trajectory, served users)."""
    obs, _ = env.reset(seed=seed)
    served, total_r, data = [], 0.0, 0.0
    done = False
    while not done:
        m, cont, _, _ = agent.select_action(obs, deterministic=True)
        obs, r, term, trunc, info = env.step((m, cont))
        served.append(m); total_r += r; data += info["delivered_Mb"]
        done = term or trunc
    summary = {
        "return": total_r, "data_Mb": data,
        "satisfied": int(np.sum(info["demand_satisfied"])),
        "D_rem_Mb": info["D_rem_Mb"], "E_left": info["E_rem_J"],
        "dist_to_end": info["dist_to_end"],
    }
    return summary, list(env.trajectory), served

## 7. Training loop and checkpoints
`n_envs` copies of the env run side by side (same users/demands as env 0 unless resampling), so every step is one batched Aer job instead of `n_envs` jobs. Plots, CSVs and the `.pt` checkpoint are saved together every `plot_every` updates, so stopping early never leaves an old `.pt` next to newer CSVs. With `resume`, the old curves are continued.

In [ ]:
def train(cfg_train=TRAIN_CFG, resume=None):
    np.random.seed(cfg_train["seed"])
    torch.manual_seed(cfg_train["seed"])
    out_dir = cfg_train["out_dir"]
    os.makedirs(out_dir, exist_ok=True)

    cfg_sys = SystemConfig(resample_users=cfg_train["resample_users"],
                           resample_demands=cfg_train["resample_demands"])
    env = UAVTrajectoryEnv(cfg_sys, seed=cfg_train["seed"])
    agent = QuantumPPOAgent(env.state_dim, env.n_users, env.cont_dim, cfg_train)
    if resume:
        load_checkpoint(agent, env, resume)

    n_updates = math.ceil(cfg_train["n_episodes"] * cfg_sys.N / cfg_train["rollout_steps"])
    print_training_header(env, cfg_train, n_updates)

    history = {k: [] for k in EP_COLS + UPD_COLS}
    history["M"] = cfg_sys.M
    episodes, update = 0, 0
    if resume:                                   # continue the old curves, not start them again
        prev = load_results(cfg_train.get("resume_history_dir") or os.path.dirname(os.path.abspath(resume)))
        if prev["episodes"] is not None:
            for c in EP_COLS:
                if c[3:] in prev["episodes"]:
                    history[c] = prev["episodes"][c[3:]].tolist()
            episodes = len(history["ep_return"])
        if prev["updates"] is not None:
            for c in UPD_COLS:
                if c in prev["updates"]:
                    history[c] = prev["updates"][c].tolist()
            update = len(history["actor_loss"])
        print(f"[Resumed history] {episodes} episodes / {update} updates loaded; "
              f"training {cfg_train['n_episodes']} more episodes")
    target_episodes = episodes + cfg_train["n_episodes"]

    E = max(1, int(cfg_train.get("n_envs", 1)))
    envs = [env]
    for i in range(1, E):
        e_i = UAVTrajectoryEnv(SystemConfig(**cfg_sys.__dict__), seed=cfg_train["seed"] + i)
        if not cfg_sys.resample_users:
            e_i.user_xy = env.user_xy.copy()
        if not cfg_sys.resample_demands:
            e_i.D = env.D.copy()
        envs.append(e_i)
    steps_per_env = max(1, cfg_train["rollout_steps"] // E)
    obs = np.stack([e_.reset()[0] for e_ in envs])
    acc = [dict(r=0.0, data=0.0, zone=0, forced=0) for _ in range(E)]

    while episodes < target_episodes:
        update += 1
        buf = {k: [] for k in ["obs", "users", "conts", "rewards", "dones", "logp", "values"]}
        n_ep_before = episodes

        for _ in range(steps_per_env):
            m, cont, logp, value = agent.select_actions(obs)
            rewards, dones, next_obs = np.zeros(E), np.zeros(E), np.empty_like(obs)
            for i, e_ in enumerate(envs):
                nobs, reward, term, trunc, info = e_.step((int(m[i]), cont[i]))
                done = term or trunc
                rewards[i], dones[i] = reward, float(done)
                a_ = acc[i]
                a_["r"] += reward; a_["data"] += info["delivered_Mb"]
                a_["zone"] += info["zone_hits"]; a_["forced"] += int(info["forced_return"])
                if done:
                    history["ep_return"].append(a_["r"])
                    history["ep_data_Mb"].append(a_["data"])
                    history["ep_satisfied"].append(int(np.sum(info["demand_satisfied"])))
                    history["ep_zone"].append(a_["zone"])
                    history["ep_forced"].append(a_["forced"])
                    history["ep_E_left"].append(info["E_rem_J"])
                    history["ep_dist_end"].append(info["dist_to_end"])
                    history["ep_update"].append(update)
                    history["ep_unmet_frac"].append(info["unmet_fraction"])
                    acc[i] = dict(r=0.0, data=0.0, zone=0, forced=0)
                    episodes += 1
                    nobs, _ = e_.reset()
                next_obs[i] = nobs
            buf["obs"].append(obs); buf["users"].append(m); buf["conts"].append(cont)
            buf["rewards"].append(rewards); buf["dones"].append(dones)
            buf["logp"].append(logp); buf["values"].append(value)
            obs = next_obs

        # bootstrap the episodes still running at the end of the rollout
        last_value = agent.values_of(obs) * (1.0 - buf["dones"][-1])
        m_stats = agent.update(buf, last_value)
        for k, v in m_stats.items():
            history[k].append(v)

        new_eps = max(1, episodes - n_ep_before)
        w = cfg_train["avg_window"]
        ret_recent = np.mean(history["ep_return"][-new_eps:]) if history["ep_return"] else float("nan")
        ma = np.mean(history["ep_return"][-w:]) if history["ep_return"] else float("nan")
        print(f" {update:>4} | {episodes:>5} | {ret_recent:>8.3f} | {ma:>8.3f} | "
              f"{np.mean(history['ep_data_Mb'][-new_eps:]):>8.1f} | "
              f"{np.mean(history['ep_satisfied'][-new_eps:]):>4.1f} | "
              f"{np.mean(history['ep_zone'][-new_eps:]):>4.1f} | "
              f"{np.mean(history['ep_E_left'][-new_eps:]):>6.0f} | "
              f"{m_stats['actor_loss']:>7.3f} | {m_stats['critic_loss']:>7.3f} | "
              f"{m_stats['entropy_user']:>5.2f} | {m_stats['kl']:>6.4f} | "
              f"{m_stats['actor_theta_grad']:>5.3f} | {m_stats['value_theta_grad']:>5.3f}")

        if update % cfg_train["plot_every"] == 0:
            eval_env = UAVTrajectoryEnv(cfg_sys, seed=cfg_train["seed"])   # same users, separate env
            eval_env.user_xy, eval_env.D = env.user_xy.copy(), env.D.copy()
            ev_summary, traj, served = evaluate(agent, eval_env)
            save_results(history, out_dir, cfg_train, cfg_sys, eval_env, ev_summary, traj, served)
            save_checkpoint(agent, env, cfg_train, cfg_sys, out_dir)
            plot_training(history, out_dir)
            plot_trajectory(eval_env, traj, served, out_dir)

    print_training_summary(history)
    plot_training(history, out_dir)
    print(f"[Saved] plots in {os.path.abspath(out_dir)}")

    summary, traj, served = evaluate(agent, env)
    print(f"[Eval greedy] return={summary['return']:.3f} | data={summary['data_Mb']:.1f} Mb | "
          f"satisfied={summary['satisfied']}/{cfg_sys.M} | E_left={summary['E_left']:.0f} J | "
          f"dist_to_end={summary['dist_to_end']:.2f} m")
    plot_trajectory(env, traj, served, out_dir)
    save_results(history, out_dir, cfg_train, cfg_sys, env, summary, traj, served)
    print(f"[Saved] data (episodes.csv, updates.csv, eval_trajectory.csv, run_info.json) in {os.path.abspath(out_dir)}")

    ckpt = save_checkpoint(agent, env, cfg_train, cfg_sys, out_dir)
    print(f"[Saved] {ckpt}")
    return agent, history

In [ ]:
def save_checkpoint(agent, env, cfg_train, cfg_sys, out_dir):
    ckpt = os.path.join(out_dir, "qppo_uav_traj_model.pt")
    tmp = ckpt + ".tmp"
    torch.save({
        "actor": agent.actor.state_dict(),
        "value": agent.value.state_dict(),
        "ret_rms": (agent.ret_rms.mean, agent.ret_rms.var, agent.ret_rms.count),
        "obs_dim": env.state_dim, "n_users": env.n_users, "cont_dim": env.cont_dim,
        "user_xy": env.user_xy, "D": env.D,
        "train_cfg": cfg_train, "sys_cfg": cfg_sys.__dict__,
        "optim": {k: getattr(agent, k).state_dict() for k in
                  ["actor_nn_opt", "actor_q_opt", "value_nn_opt", "value_q_opt"]},
    }, tmp)
    os.replace(tmp, ckpt)                     # atomic: a crash mid-save can't corrupt the old .pt
    return ckpt


def load_checkpoint(agent, env, path):
    """Continue from a saved .pt: networks, PQC parameters, return scaling, same users/demands."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if ck["obs_dim"] != env.state_dim or ck["n_users"] != env.n_users:
        raise ValueError(f"checkpoint was trained with obs_dim={ck['obs_dim']}, M={ck['n_users']}, "
                         f"but the current config has obs_dim={env.state_dim}, M={env.n_users}")
    agent.actor.load_state_dict(ck["actor"])
    agent.value.load_state_dict(ck["value"])
    agent.ret_rms.mean, agent.ret_rms.var, agent.ret_rms.count = ck["ret_rms"]
    for k, sd in ck.get("optim", {}).items():          # older checkpoints have no optimizer state
        getattr(agent, k).load_state_dict(sd)
    if not env.cfg.resample_users:
        env.user_xy = np.asarray(ck["user_xy"])
    if not env.cfg.resample_demands:
        env.D = np.asarray(ck["D"])
    print(f"[Resumed] {path}  (optimizer state {'restored' if 'optim' in ck else 'reset'})")

## 8. Run settings

This replaces the script's command-line arguments.

| script | here |
|---|---|
| `python training_quantumPPO_fast.py` | defaults below |
| `--noise` | `NOISE = True` |
| `--noise --noise-backend FakeManilaV2` | `NOISE = True`, `NOISE_BACKEND = "FakeManilaV2"` |
| `--resume results_hw/qppo_uav_traj_model.pt` | `RESUME = "results_hw/qppo_uav_traj_model.pt"` |

In [ ]:
NOISE            = False                       # noisy Aer simulation
SHOTS            = TRAIN_CFG["shots"]          # shots per circuit when NOISE
NOISE_BACKEND    = TRAIN_CFG["noise_backend"]  # e.g. "FakeManilaV2"
N_EPISODES       = TRAIN_CFG["n_episodes"]
ROLLOUT_STEPS    = TRAIN_CFG["rollout_steps"]
RESAMPLE_USERS   = False                       # new user positions every episode
RESAMPLE_DEMANDS = False                       # new demands every episode
SEED             = TRAIN_CFG["seed"]
OUT_DIR          = TRAIN_CFG["out_dir"]
RESUME           = None                        # path to a saved .pt to continue from
HISTORY_DIR      = None                        # folder of the old CSVs (default: the .pt's folder)

cfg = dict(TRAIN_CFG)
cfg.update(noise=NOISE, shots=SHOTS, noise_backend=NOISE_BACKEND,
           n_episodes=N_EPISODES, rollout_steps=ROLLOUT_STEPS,
           resample_users=RESAMPLE_USERS, resample_demands=RESAMPLE_DEMANDS,
           seed=SEED, out_dir=OUT_DIR, resume_history_dir=HISTORY_DIR)

### Train
Prints one log line per PPO update and saves plots/results every `plot_every` updates.

In [ ]:
agent, history = train(cfg, resume=RESUME)

## 9. Results
Shows the plots saved in `OUT_DIR`. Works on its own after a restart (only sections 1 and 6 need to have run), so you can reopen old runs.

In [ ]:
from IPython.display import Image, display
for name in ["quantum_ppo_training.png", "trajectory.png"]:
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(name); display(Image(filename=path))
    else:
        print(f"[missing] {path}")

In [ ]:
run = load_results(OUT_DIR)
print(run["info"].get("algorithm"), "|", run["info"].get("n_episodes_done"), "episodes |",
      run["info"].get("n_updates_done"), "updates")
print("greedy eval:", json.dumps(run["info"].get("eval_greedy"), indent=2))